In [ ]:
import marimo as mo
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    roc_auc_score,
)
from xgboost import plot_tree
import xgboost as xgb
import matplotlib.pyplot as plt
import shap 
import seaborn as sns

In [ ]:
junction_path = '../data/3way_junctions_labeled.csv'
df = pd.read_csv(junction_path)
df = df.dropna()
df

[delta energy for 0 length](https://rna.urmc.rochester.edu/NNDB/parameter_tables/turner_2004_rnastructure/turner_2004_watson_crick_stack_dg.txt)

[delta energy for 1 length](https://rna.urmc.rochester.edu/NNDB/parameter_tables/turner_2004_rnastructure/turner_2004_terminal_mismatch_dg.txt)

In [ ]:
FAMILY = {'A':0,'B':1, 'C':2}
# Terminal mismatch energy values from Turner 2004 parameters
# where closing_pair is the Watson-Crick pair and mismatch_pair is X,Y from the 5'->3' direction

energy_length1 = {
    # AU/UA closing pair with terminal mismatches
    ('AU', 'AA'): -0.8,
    ('AU', 'AC'): -1.0,
    ('AU', 'AG'): -0.8,
    ('AU', 'AU'): -1.0,
    ('AU', 'CA'): -0.6,
    ('AU', 'CC'): -0.7,
    ('AU', 'CG'): -0.6,
    ('AU', 'CU'): -0.7,
    ('AU', 'GA'): -0.8,
    ('AU', 'GC'): -1.0,
    ('AU', 'GG'): -0.8,
    ('AU', 'GU'): -1.0,
    ('AU', 'UA'): -0.6,
    ('AU', 'UC'): -0.8,
    ('AU', 'UG'): -0.6,
    ('AU', 'UU'): -0.8,

    # CG/GC closing pair with terminal mismatches
    ('CG', 'AA'): -1.5,
    ('CG', 'AC'): -1.5,
    ('CG', 'AG'): -1.4,
    ('CG', 'AU'): -1.5,
    ('CG', 'CA'): -1.0,
    ('CG', 'CC'): -1.1,
    ('CG', 'CG'): -1.0,
    ('CG', 'CU'): -0.8,
    ('CG', 'GA'): -1.4,
    ('CG', 'GC'): -1.5,
    ('CG', 'GG'): -1.6,
    ('CG', 'GU'): -1.5,
    ('CG', 'UA'): -1.0,
    ('CG', 'UC'): -1.4,
    ('CG', 'UG'): -1.0,
    ('CG', 'UU'): -1.2,

    # GC/CG closing pair with terminal mismatches
    ('GC', 'AA'): -1.1,
    ('GC', 'AC'): -1.5,
    ('GC', 'AG'): -1.3,
    ('GC', 'AU'): -1.5,
    ('GC', 'CA'): -1.1,
    ('GC', 'CC'): -0.7,
    ('GC', 'CG'): -1.1,
    ('GC', 'CU'): -0.5,
    ('GC', 'GA'): -1.6,
    ('GC', 'GC'): -1.5,
    ('GC', 'GG'): -1.4,
    ('GC', 'GU'): -1.5,
    ('GC', 'UA'): -1.1,
    ('GC', 'UC'): -1.0,
    ('GC', 'UG'): -1.1,
    ('GC', 'UU'): -0.7,

    # GU/UG closing pair with terminal mismatches
    ('GU', 'AA'): -0.3,
    ('GU', 'AC'): -1.0,
    ('GU', 'AG'): -0.8,
    ('GU', 'AU'): -1.0,
    ('GU', 'CA'): -0.6,
    ('GU', 'CC'): -0.7,
    ('GU', 'CG'): -0.6,
    ('GU', 'CU'): -0.7,
    ('GU', 'GA'): -0.6,
    ('GU', 'GC'): -1.0,
    ('GU', 'GG'): -0.8,
    ('GU', 'GU'): -1.0,
    ('GU', 'UA'): -0.6,
    ('GU', 'UC'): -0.8,
    ('GU', 'UG'): -0.6,
    ('GU', 'UU'): -0.6,

    # UA/AU closing pair with terminal mismatches
    ('UA', 'AA'): -1.0,
    ('UA', 'AC'): -0.8,
    ('UA', 'AG'): -1.1,
    ('UA', 'AU'): -0.8,
    ('UA', 'CA'): -0.7,
    ('UA', 'CC'): -0.6,
    ('UA', 'CG'): -0.7,
    ('UA', 'CU'): -0.5,
    ('UA', 'GA'): -1.1,
    ('UA', 'GC'): -0.8,
    ('UA', 'GG'): -1.2,
    ('UA', 'GU'): -0.8,
    ('UA', 'UA'): -0.7,
    ('UA', 'UC'): -0.6,
    ('UA', 'UG'): -0.7,
    ('UA', 'UU'): -0.5,

    # UG/GU closing pair with terminal mismatches
    ('UG', 'AA'): -1.0,
    ('UG', 'AC'): -0.8,
    ('UG', 'AG'): -1.1,
    ('UG', 'AU'): -0.8,
    ('UG', 'CA'): -0.7,
    ('UG', 'CC'): -0.6,
    ('UG', 'CG'): -0.7,
    ('UG', 'CU'): -0.5,
    ('UG', 'GA'): -0.5,
    ('UG', 'GC'): -0.8,
    ('UG', 'GG'): -0.8,
    ('UG', 'GU'): -0.8,
    ('UG', 'UA'): -0.7,
    ('UG', 'UC'): -0.6,
    ('UG', 'UG'): -0.7,
    ('UG', 'UU'): -0.5
}

energy_length_no_unpaired_nt = energy_values = {
    # AU/UA table
    ('AU', 'UA'): -0.9,
    ('AU', 'CG'): -2.2,
    ('AU', 'GC'): -2.1, 
    ('AU', 'GU'): -0.6,
    ('AU', 'UA'): -1.1,
    ('AU', 'UG'): -1.4,

    # CG/GC table
    ('CG', 'UA'): -2.1,
    ('CG', 'CG'): -3.3,
    ('CG', 'GC'): -2.4,
    ('CG', 'GU'): -1.4,
    ('CG', 'UA'): -2.1,
    ('CG', 'UG'): -2.1,

    # GC/CG table
    ('GC', 'UA'): -2.4,
    ('GC', 'CG'): -3.4,
    ('GC', 'GC'): -3.3,
    ('GC', 'GU'): -1.5,
    ('GC', 'UA'): -2.2,
    ('GC', 'UG'): -2.5,

    # GU/UG table
    ('GU', 'UA'): -1.3,
    ('GU', 'CG'): -2.5,
    ('GU', 'GC'): -2.1,
    ('GU', 'GU'): -0.5,
    ('GU', 'UA'): -1.4,
    ('GU', 'UG'): 1.3,  # Note: this is positive

    # UA/AU table
    ('UA', 'UA'): -1.3,
    ('UA', 'CG'): -2.4,
    ('UA', 'GC'): -2.1,
    ('UA', 'GU'): -1.0,
    ('UA', 'UA'): -0.9,
    ('UA', 'UG'): -1.3,

    # UG/GU table
    ('UG', 'UA'): -1.0,
    ('UG', 'CG'): -1.5,
    ('UG', 'GC'): -1.4,
    ('UG', 'GU'): 0.3,  # Note: this is positive
    ('UG', 'UA'): -0.6,
    ('UG', 'UG'): -0.5
}

In [ ]:
def consecutive_adenines(sequence: str) -> int:
    nt = {'A': [0], 'C': [0], 'G': [0], 'U': [0]}
    temp = 0
    curr = sequence[1]
    for seq in sequence[1:len(sequence)-1]:
        if seq == curr:
            temp = temp + 1
        else:
            nt[curr].append(temp)
            curr = seq
            temp = 1
    nt[curr].append(temp)
    a = sorted(nt['A'])[-1]
    if a > 1:
        return a
    else:
        return 0

In [ ]:
def calculate_energy(length:int,seq1:str, seq2:str, seq3:str) -> float:
    a= 9.3
    b = -0.3
    c = -0.9
    h = 2
    if length > 6:
        return round(a+ 6*b+1.1* np.log(length/6) + c*h, 1)
    elif length >= 2 and length <= 6:
        return round(a+ length*b + c*h, 1)
    elif length ==1:
        bps1 = seq1[0]+seq3[-1]
        bps2 = seq1[-1]+seq2[0]
        # return (bps1,bps2)
        return round(energy_length1.get((bps1,bps2), (bps2,bps1)) +2.1,1)
    else:
        bps = (seq1[0]+seq3[-1],seq1[-1]+seq2[0])
        return energy_length_no_unpaired_nt.get(bps,0)

In [ ]:
def sort_values(n1:int,n2:int,n3:int,idx:int) -> int:
    sort = sorted([n1,n2,n3])
    return sort[idx]

In [ ]:
df['ΔG(H1,H2)'] = df.apply(lambda x: calculate_energy(x['|J12|'], x['J12'], x['J23'],x['J31']), axis=1)
df['ΔG(H2,H3)'] = df.apply(lambda x: calculate_energy(x['|J23|'], x['J23'], x['J31'],x['J12']), axis=1)
df['ΔG(H3,H1)'] = df.apply(lambda x: calculate_energy(x['|J31|'], x['J31'], x['J12'],x['J23']), axis=1)
df['A(J12)'] = df['J12'].apply(consecutive_adenines)
df['A(J23)'] = df['J23'].apply(consecutive_adenines)
df['A(J31)'] = df['J31'].apply(consecutive_adenines)

df['minJ23'] = df[['|J12|','|J31|']].min(axis=1)
df['minJ31'] = df[['|J12|','|J23|']].min(axis=1)
df['minJ12'] = df[['|J23|','|J31|']].min(axis=1)
df['sort|J12|'] = df[['|J12|','|J23|','|J31|']].apply(lambda row: sort_values(row.iloc[0], row.iloc[1], row.iloc[2], 0), axis=1)
df['sort|J23|'] = df[['|J12|','|J23|','|J31|']].apply(lambda row: sort_values(row.iloc[0], row.iloc[1], row.iloc[2], 1), axis=1)
df['sort|J31|'] = df[['|J12|','|J23|','|J31|']].apply(lambda row: sort_values(row.iloc[0], row.iloc[1], row.iloc[2], 2), axis=1)

In [ ]:
X = df.drop(columns=['Junction_ID', 'Family', 'Coaxial','J12','J23','J31',
                          'J12_ID5','J23_ID5','J31_ID5','J12_ID3','J23_ID3','J31_ID3'])
features = list(X.columns)
y = df['Family'].map(FAMILY)